# XR Client Utilities

Examples of defining user commands that can be invoked by the XR client's command menu.

<video controls src="assets/xr-client-utilities.mp4" alt="video demonstrating the commands in xr" />

## Setup runner

In [1]:
from nanover.app import OmniRunner
from nanover.openmm import OpenMMSimulation

simulation = OpenMMSimulation.from_xml_path("../openmm/openmm_files/17-ala.xml")
simulation.load()

imd_runner = OmniRunner.with_basic_server(simulation, port=0, name="xr client utilities example")
imd_runner.load(0)
imd = imd_runner.app_server.imd

## Prepare universe
We use this to get atom data more easily for the "info mode" later.

In [2]:
from nanover.mdanalysis import frame_data_to_mdanalysis

universe = frame_data_to_mdanalysis(simulation.make_topology_frame())

## XR client notification
The XR client defines a remote command to send it notifications (displayed over the controller). We define a function to find all of them and call them to notify all clients.

In [3]:
def notify_all(message):
    for command in imd_runner.app_server.commands:
        if "/notify" in command:
            imd_runner.app_server.run_command(command, dict(message=message))

## Modes & interaction triggers
This code snippet defines a basic system of "modes" that can be switched between by the user, providing different sets of behavior in response to interactions starting and stopping.

In [4]:
from nanover.imd.imd_state import ParticleInteraction

class Mode:
    active = None

    @staticmethod
    def add_to_runner(imd_runner):
        def on_interaction_started(*, key: str, interaction: ParticleInteraction):
            Mode.active.active.on_interaction_started(key=key, interaction=interaction)

        def on_interaction_stopped(*, key: str, interaction: ParticleInteraction):
            Mode.active.on_interaction_stopped(key=key, interaction=interaction)

        imd_runner.app_server.imd.interaction_started.add_callback(on_interaction_started)
        imd_runner.app_server.imd.interaction_stopped.add_callback(on_interaction_stopped)
        Mode.enter()

    @classmethod
    def enter(cls):
        Mode.active = cls()
        notify_all(f"ENTERED {cls.__name__}")

    def on_interaction_started(self, *, key: str, interaction: ParticleInteraction):
        pass

    def on_interaction_stopped(self, *, key: str, interaction: ParticleInteraction):
        pass

Mode.add_to_runner(imd_runner)

In [5]:
class NormalMode(Mode):
    pass

imd_runner.app_server.register_command("user/normal mode", NormalMode.enter)

## Info mode
Defining a mode where every user interaction triggers a notification in the XR client with information about the interacted atom from MDAnalysis:

In [6]:
class InfoMode(Mode):
    def on_interaction_started(self, *, key: str, interaction: ParticleInteraction):
        atom = universe.atoms[interaction.particles[0]]
        notify_all(str(atom))

imd_runner.app_server.register_command("user/info mode", InfoMode.enter)

## Recording
Defining commands that start and stop recording to a file:

In [7]:
from nanover.websocket.record import record_from_runner, BackgroundRecordingContext

RECORDING_PATH = None
RECORDING_COUNT = 0
RECORDER: BackgroundRecordingContext | None = None

def start_recording():
    global RECORDER, RECORDING_COUNT, RECORDING_PATH
    RECORDING_PATH = f"RECORDING-{RECORDING_COUNT}-{imd_runner.simulation.name}.nanover.zip"
    RECORDER = record_from_runner(imd_runner, RECORDING_PATH)
    RECORDING_COUNT += 1
    notify_all(f"STARTED RECORDING to {RECORDING_PATH}")

def stop_recording():
    if RECORDER is not None:
        RECORDER.close()
        notify_all(f"FINISHED RECORDING to {RECORDING_PATH}")

imd_runner.app_server.register_command("user/recording/start", start_recording)
imd_runner.app_server.register_command("user/recording/stop", stop_recording)